# ONE ALL MODELS Dataset — disk fix v2

Use **Image_Studio_All_Models_Dataset_v2.ipynb** on CPU with Internet. This creates exactly **ONE private Dataset containing all eight models**. No model Dataset inputs are required, and Studio v9 remains unchanged: attach that ONE completed Dataset there. Its 24 workflows remain in the studio notebook.

The original combined builder incorrectly required a complete 20.53 GB local output before upload. This creator instead reuses verified files directly from `/kaggle/input` and earlier local pack outputs. It never copies, moves or deletes those existing sources. A missing model is downloaded to a dedicated temporary cache, verified by SHA256, passed to Kaggle's normal uploader, and released only after a file-upload token is recorded. The native creation call then submits all uploaded file tokens together as ONE private Dataset. Metadata and original license notices are included.

**Disk needed: the largest missing model plus 512 MiB reserve**, rather than the sum of all models. With no reusable files, the largest model is about 8.05 GB; with saved files available it can be smaller. No full second copy or zip archive is made. If all models already exist, only a small metadata reserve is needed. The completed Dataset still contains the full 20.53 GB / 19.12 GiB of model bytes; Kaggle's account Dataset quotas still apply.

Authentication and the final `api.dataset_create_new(public=False)` call use the working v3 method. There is no added datasets.get prerequisite. To bound disk usage, the SDK's uploader receives one model folder at a time within the same creation request and upload context. Original Kaggle 2.0.2 creation/uploader method bodies are exercised by offline regression tests. No forced SDK upgrade is used.

Keep the session running until **PRIVATE ALL MODELS DATASET SUBMITTED** appears. Wait for Kaggle processing, then attach that Dataset to Studio v9. Notebook outputs are **not** a full model-pack backup in this mode because successfully uploaded temporary downloads have been released. Existing saved inputs and prior pack directories remain untouched. If final creation fails, the error is retained; never treat per-file upload messages alone as confirmed Dataset creation. A saved success receipt skips resubmission. An interrupted retry may need to download temporary files already released after upload.

This is a CPU storage/upload correction only. Studio v9 does not prepare/download weights, and no split Dataset inputs are introduced. Actual authenticated publication of the complete pack remains unverified in this development environment.


In [ ]:
# CPU-only model pack builder. Run once and save the outputs on Kaggle.
PACK_BUILDER_VERSION = 'dataset-upload-v3-direct-create'
ALL_MODELS_STORAGE_VERSION = 'single-model-cache-v2'
AUTO_CREATE_DATASET = True  # Uses your existing Kaggle authentication to create a private Dataset.
DATASET_OWNER = ''  # Usually detected from authenticated Kaggle API credentials.
PACK = 'all'  # All eight reviewed models in one private Dataset (~20.53 GB / 19.12 GiB)
from pathlib import Path
import os, json, hashlib, shutil, time, urllib.request, urllib.error, fcntl
CATALOG = {'profile': 'commercial-v7', 'reviewed': '2026-11-09', 'assets': [{'pack': 'create_edit', 'folder': 'diffusion_models', 'name': 'flux-2-klein-4b.safetensors', 'bytes': 7751105712, 'sha256': 'ec3d4e733a771f61c052fb4856c48b336c55eaf2c65487c2a1faeb9bbda7a343', 'license': 'Apache-2.0', 'source': 'https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/5b3abca5c0dc1ecd4eaf3340868d09063fc1c965/split_files/diffusion_models/flux-2-klein-4b.safetensors'}, {'pack': 'create_edit', 'folder': 'text_encoders', 'name': 'qwen_3_4b.safetensors', 'bytes': 8044982048, 'sha256': '6c671498573ac2f7a5501502ccce8d2b08ea6ca2f661c458e708f36b36edfc5a', 'license': 'Apache-2.0', 'source': 'https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/5b3abca5c0dc1ecd4eaf3340868d09063fc1c965/split_files/text_encoders/qwen_3_4b.safetensors'}, {'pack': 'create_edit', 'folder': 'vae', 'name': 'flux2-vae.safetensors', 'bytes': 336211292, 'sha256': '868fe7b343cc8f3a19dbcfcafbc3d5f888802be3f89bd81b65b3621a066ce8f3', 'license': 'Apache-2.0', 'source': 'https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/5b3abca5c0dc1ecd4eaf3340868d09063fc1c965/split_files/vae/flux2-vae.safetensors'}, {'pack': 'finish', 'folder': 'upscale_models', 'name': 'RealESRGAN_x4plus.pth', 'bytes': 67040989, 'sha256': '4fa0d38905f75ac06eb49a7951b426670021be3018265fd191d2125df9d682f1', 'license': 'BSD-3-Clause', 'source': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'}, {'pack': 'finish', 'folder': 'diffusion_models', 'name': 'seedvr2_3b_int8_convrot.safetensors', 'bytes': 3458259704, 'sha256': 'c3dec8bcc5916843a8a858572970597462e1f2dc598d6dfd818f6cd40f53a157', 'license': 'Apache-2.0', 'source': 'https://huggingface.co/Comfy-Org/SeedVR2/resolve/df48879708206a403d2a61acd55578c2e80fd233/diffusion_models/seedvr2_3b_int8_convrot.safetensors'}, {'pack': 'finish', 'folder': 'vae', 'name': 'seedvr2_ema_vae_fp16.safetensors', 'bytes': 501324814, 'sha256': '20678548f420d98d26f11442d3528f8b8c94e57ee046ef93dbb7633da8612ca1', 'license': 'Apache-2.0', 'source': 'https://huggingface.co/Comfy-Org/SeedVR2/resolve/df48879708206a403d2a61acd55578c2e80fd233/vae/seedvr2_ema_vae_fp16.safetensors'}, {'pack': 'finish', 'folder': 'mediapipe', 'name': 'face_landmarker.task', 'bytes': 3758596, 'sha256': '64184e229b263107bc2b804c6625db1341ff2bb731874b0bcc2fe6544e0bc9ff', 'license': 'Apache-2.0', 'source': 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task'}, {'pack': 'finish', 'folder': 'loras', 'name': 'bfs_head_v1_flux-klein_4b.safetensors', 'bytes': 369644384, 'sha256': '77aabcd6144612dd4873080b99283dd9a27977712548bee161894f62c91ff09c', 'license': 'MIT', 'source': 'https://huggingface.co/Alissonerdx/BFS-Best-Face-Swap/resolve/2209ae537ac5e883d6ec7db5b7bd6dccf1a0eba7/bfs_head_v1_flux-klein_4b.safetensors'}]}
LICENSE_FILES = {'realesrgan-BSD.txt': 'BSD 3-Clause License\n\nCopyright (c) 2021, Xintao Wang\nAll rights reserved.\n\nRedistribution and use in source and binary forms, with or without\nmodification, are permitted provided that the following conditions are met:\n\n1. Redistributions of source code must retain the above copyright notice, this\n   list of conditions and the following disclaimer.\n\n2. Redistributions in binary form must reproduce the above copyright notice,\n   this list of conditions and the following disclaimer in the documentation\n   and/or other materials provided with the distribution.\n\n3. Neither the name of the copyright holder nor the names of its\n   contributors may be used to endorse or promote products derived from\n   this software without specific prior written permission.\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"\nAND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE\nIMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\nDISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE\nFOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL\nDAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR\nSERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER\nCAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,\nOR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE\nOF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n', 'mediapipe-model-evidence.txt': 'MediaPipe model license evidence, reviewed 2026-11-09. Bundle SHA256: 64184e229b263107bc2b804c6625db1341ff2bb731874b0bcc2fe6544e0bc9ff\n\nhttps://storage.googleapis.com/mediapipe-assets/Model%20Card%20MediaPipe%20Face%20Mesh%20V2.pdf\nGeng Yan, Google\nIvan Grishchenko, Google\nMODEL DATE\nSeptember 15, 2022\nLICENSED UNDER\nApache License, Version 2.0\n\n_\nBased on Model Cards for Model Repo ing, In Proceedings of FAT* Conference (FAT*2019). ACM, New York, NY, USA\n\nhttps://storage.googleapis.com/mediapipe-assets/MediaPipe%20BlazeFace%20Model%20Card%20(Short%20Range).pdf\n\nDOCUMENTATION\nPaper: https://arxiv.org/abs/2006.10204\n\nLICENSED UNDER\nApache License, Version 2.0\n\nBased on Model Cards for Model Reporting, In Proceedings of FAT* Conference (FAT*2019). ACM, New York, NY, USA\n\n\nhttps://storage.googleapis.com/mediapipe-assets/Model%20Card%20Blendshape%20V2.pdf\nAppendix.\n\nfront-facing camera\n\nLICENSED UNDER\nApache License, Version 2.0\nLe : Input frame with predicted facial landmarks.\nRight: 3D avatar controlled with predicted blendshape\ncoe cients\n', 'flux2-klein4b-APACHE.txt': '                             Apache License\n                       Version 2.0, January 2004\n                    http://www.apache.org/licenses/\n\nTERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n1. Definitions.\n\n  "License" shall mean the terms and conditions for use, reproduction,\n  and distribution as defined by Sections 1 through 9 of this document.\n\n  "Licensor" shall mean the copyright owner or entity authorized by\n  the copyright owner that is granting the License.\n\n  "Legal Entity" shall mean the union of the acting entity and all\n  other entities that control, are controlled by, or are under common\n  control with that entity. For the purposes of this definition,\n  "control" means (i) the power, direct or indirect, to cause the\n  direction or management of such entity, whether by contract or\n  otherwise, or (ii) ownership of fifty percent (50%) or more of the\n  outstanding shares, or (iii) beneficial ownership of such entity.\n\n  "You" (or "Your") shall mean an individual or Legal Entity\n  exercising permissions granted by this License.\n\n  "Source" form shall mean the preferred form for making modifications,\n  including but not limited to software source code, documentation\n  source, and configuration files.\n\n  "Object" form shall mean any form resulting from mechanical\n  transformation or translation of a Source form, including but\n  not limited to compiled object code, generated documentation,\n  and conversions to other media types.\n\n  "Work" shall mean the work of authorship, whether in Source or\n  Object form, made available under the License, as indicated by a\n  copyright notice that is included in or attached to the work\n  (an example is provided in the Appendix below).\n\n  "Derivative Works" shall mean any work, whether in Source or Object\n  form, that is based on (or derived from) the Work and for which the\n  editorial revisions, annotations, elaborations, or other modifications\n  represent, as a whole, an original work of authorship. For the purposes\n  of this License, Derivative Works shall not include works that remain\n  separable from, or merely link (or bind by name) to the interfaces of,\n  the Work and Derivative Works thereof.\n\n  "Contribution" shall mean any work of authorship, including\n  the original version of the Work and any modifications or additions\n  to that Work or Derivative Works thereof, that is intentionally\n  submitted to Licensor for inclusion in the Work by the copyright owner\n  or by an individual or Legal Entity authorized to submit on behalf of\n  the copyright owner. For the purposes of this definition, "submitted"\n  means any form of electronic, verbal, or written communication sent\n  to the Licensor or its representatives, including but not limited to\n  communication on electronic mailing lists, source code control systems,\n  and issue tracking systems that are managed by, or on behalf of, the\n  Licensor for the purpose of discussing and improving the Work, but\n  excluding communication that is conspicuously marked or otherwise\n  designated in writing by the copyright owner as "Not a Contribution."\n\n  "Contributor" shall mean Licensor and any individual or Legal Entity\n  on behalf of whom a Contribution has been received by Licensor and\n  subsequently incorporated within the Work.\n\n2. Grant of Copyright License. Subject to the terms and conditions of\n  this License, each Contributor hereby grants to You a perpetual,\n  worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n  copyright license to reproduce, prepare Derivative Works of,\n  publicly display, publicly perform, sublicense, and distribute the\n  Work and such Derivative Works in Source or Object form.\n\n3. Grant of Patent License. Subject to the terms and conditions of\n  this License, each Contributor hereby grants to You a perpetual,\n  worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n  (except as stated in this section) patent license to make, have made,\n  use, offer to sell, sell, import, and otherwise transfer the Work,\n  where such license applies only to those patent claims licensable\n  by such Contributor that are necessarily infringed by their\n  Contribution(s) alone or by combination of their Contribution(s)\n  with the Work to which such Contribution(s) was submitted. If You\n  institute patent litigation against any entity (including a\n  cross-claim or counterclaim in a lawsuit) alleging that the Work\n  or a Contribution incorporated within the Work constitutes direct\n  or contributory patent infringement, then any patent licenses\n  granted to You under this License for that Work shall terminate\n  as of the date such litigation is filed.\n\n4. Redistribution. You may reproduce and distribute copies of the\n  Work or Derivative Works thereof in any medium, with or without\n  modifications, and in Source or Object form, provided that You\n  meet the following conditions:\n\n  (a) You must give any other recipients of the Work or\n      Derivative Works a copy of this License; and\n\n  (b) You must cause any modified files to carry prominent notices\n      stating that You changed the files; and\n\n  (c) You must retain, in the Source form of any Derivative Works\n      that You distribute, all copyright, patent, trademark, and\n      attribution notices from the Source form of the Work,\n      excluding those notices that do not pertain to any part of\n      the Derivative Works; and\n\n  (d) If the Work includes a "NOTICE" text file as part of its\n      distribution, then any Derivative Works that You distribute must\n      include a readable copy of the attribution notices contained\n      within such NOTICE file, excluding those notices that do not\n      pertain to any part of the Derivative Works, in at least one\n      of the following places: within a NOTICE text file distributed\n      as part of the Derivative Works; within the Source form or\n      documentation, if provided along with the Derivative Works; or,\n      within a display generated by the Derivative Works, if and\n      wherever such third-party notices normally appear. The contents\n      of the NOTICE file are for informational purposes only and\n      do not modify the License. You may add Your own attribution\n      notices within Derivative Works that You distribute, alongside\n      or as an addendum to the NOTICE text from the Work, provided\n      that such additional attribution notices cannot be construed\n      as modifying the License.\n\n  You may add Your own copyright statement to Your modifications and\n  may provide additional or different license terms and conditions\n  for use, reproduction, or distribution of Your modifications, or\n  for any such Derivative Works as a whole, provided Your use,\n  reproduction, and distribution of the Work otherwise complies with\n  the conditions stated in this License.\n\n5. Submission of Contributions. Unless You explicitly state otherwise,\n  any Contribution intentionally submitted for inclusion in the Work\n  by You to the Licensor shall be under the terms and conditions of\n  this License, without any additional terms or conditions.\n  Notwithstanding the above, nothing herein shall supersede or modify\n  the terms of any separate license agreement you may have executed\n  with Licensor regarding such Contributions.\n\n6. Trademarks. This License does not grant permission to use the trade\n  names, trademarks, service marks, or product names of the Licensor,\n  except as required for reasonable and customary use in describing the\n  origin of the Work and reproducing the content of the NOTICE file.\n\n7. Disclaimer of Warranty. Unless required by applicable law or\n  agreed to in writing, Licensor provides the Work (and each\n  Contributor provides its Contributions) on an "AS IS" BASIS,\n  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n  implied, including, without limitation, any warranties or conditions\n  of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n  PARTICULAR PURPOSE. You are solely responsible for determining the\n  appropriateness of using or redistributing the Work and assume any\n  risks associated with Your exercise of permissions under this License.\n\n8. Limitation of Liability. In no event and under no legal theory,\n  whether in tort (including negligence), contract, or otherwise,\n  unless required by applicable law (such as deliberate and grossly\n  negligent acts) or agreed to in writing, shall any Contributor be\n  liable to You for damages, including any direct, indirect, special,\n  incidental, or consequential damages of any character arising as a\n  result of this License or out of the use or inability to use the\n  Work (including but not limited to damages for loss of goodwill,\n  work stoppage, computer failure or malfunction, or any and all\n  other commercial damages or losses), even if such Contributor\n  has been advised of the possibility of such damages.\n\n9. Accepting Warranty or Additional Liability. While redistributing\n  the Work or Derivative Works thereof, You may choose to offer,\n  and charge a fee for, acceptance of support, warranty, indemnity,\n  or other liability obligations and/or rights consistent with this\n  License. However, in accepting such obligations, You may act only\n  on Your own behalf and on Your sole responsibility, not on behalf\n  of any other Contributor, and only if You agree to indemnify,\n  defend, and hold each Contributor harmless for any liability\n  incurred by, or claims asserted against, such Contributor by reason\n  of your accepting any such warranty or additional liability.\n\nEND OF TERMS AND CONDITIONS\n', 'bfs-model-card-MIT.md': 'Original model card snapshot, declared MIT. Source: https://huggingface.co/Alissonerdx/BFS-Best-Face-Swap/blob/2209ae537ac5e883d6ec7db5b7bd6dccf1a0eba7/README.md\n\n---\nlicense: mit\nbase_model:\n- Qwen/Qwen-Image-2.1\n- Qwen/Qwen-Image-Edit-2511\n- black-forest-labs/FLUX.2-klein-9B\n- black-forest-labs/FLUX.2-klein-4B\n- black-forest-labs/FLUX.2-klein-base-4B\n- black-forest-labs/FLUX.2-klein-base-9B\n- black-forest-labs/FLUX.2-klein-9b-fp8\n- black-forest-labs/FLUX.2-klein-4b-fp8\n- krea/Krea-2-Raw\npipeline_tag: image-to-image\ntags:\n- lora\n- qwen-image\n- qwen-image-2.1\n- qwen-image-edit\n- face-swap\n- head-swap\n- body-swap\n- selective-body-swap\n- image-editing\n- comfyui\n- bfs\nlanguage:\n- en\nlibrary_name: diffusers\n---\n\n# BFS: Best Face Swap\n\nBFS is a collection of LoRAs for face, head, and body replacement across Qwen Image 2.1, Qwen\nImage Edit 2509/2511, Flux 2 Klein, Krea 2, and experimental LTX-2 workflows. Each release has\nits own base-model, input-order, prompt, and workflow contract. Use the guide for the exact\nversion you are loading.\n\n## Guides\n\n- [BFS Head V1 / V1.1 for Qwen Image 2.1](docs/qwen-image-2.1.md)\n- [BFS Body Swap V1.0 for Qwen Image 2.1](docs/qwen-image-2.1-body.md)\n- [BFS Body Swap Selective (Magenta) V1.0 for Qwen Image 2.1](docs/qwen-image-2.1-body-selective.md):\n  replaces only the person marked in magenta, for group photos and busy scenes\n- [Qwen Image Edit 2509/2511](docs/qwen-image-edit.md)\n- [Flux 2 Klein](docs/flux-2-klein.md)\n- [Krea 2](docs/krea-2.md)\n- [Experimental LTX-2 weights](docs/ltx-2.md)\n\n## Downloads at a glance\n\n| Family | Weights | Workflow |\n|---|---|---|\n| Qwen Image 2.1 (head) | [`bfs_head_v1.1_qwen_2.1.safetensors`](bfs_head_v1.1_qwen_2.1.safetensors) (recommended), [`bfs_head_v1.1_alternative_qwen_2.1.safetensors`](bfs_head_v1.1_alternative_qwen_2.1.safetensors), [`bfs_head_v1_qwen_2.1.safetensors`](bfs_head_v1_qwen_2.1.safetensors) | [`Head Swap V1 Qwen 2.1 Workflow.json`](workflows/Head%20Swap%20V1%20Qwen%202.1%20Workflow.json) |\n| Qwen Image 2.1 (body) | [`bfs_body_swap_v1.0_qwen_2.1.safetensors`](bfs_body_swap_v1.0_qwen_2.1.safetensors) | [`Body Swap Qwen Image 2.1 - V1.json`](workflows/Body%20Swap%20Qwen%20Image%202.1%20-%20V1.json), see the [body swap guide](docs/qwen-image-2.1-body.md) |\n| Qwen Image 2.1 (selective body, magenta mark) | [`bfs_body_swap_selective_magenta_v1.0_qwen_2.1.safetensors`](bfs_body_swap_selective_magenta_v1.0_qwen_2.1.safetensors) | [`Body Swap Selective Qwen Image 2.1 - V1.0.json`](workflows/Body%20Swap%20Selective%20Qwen%20Image%202.1%20-%20V1.0.json), see the [selective guide](docs/qwen-image-2.1-body-selective.md) |\n| Qwen Image Edit 2509 | [`bfs_face_v1_qwen_image_edit_2509.safetensors`](bfs_face_v1_qwen_image_edit_2509.safetensors), [`bfs_head_v1_qwen_image_edit_2509.safetensors`](bfs_head_v1_qwen_image_edit_2509.safetensors), [`bfs_head_v2_qwen_image_edit_2509.safetensors`](bfs_head_v2_qwen_image_edit_2509.safetensors), [`bfs_head_v3_qwen_image_edit_2509.safetensors`](bfs_head_v3_qwen_image_edit_2509.safetensors), [`bfs_head_v4_qwen_image_edit_2509.safetensors`](bfs_head_v4_qwen_image_edit_2509.safetensors) | See the [Qwen Edit guide](docs/qwen-image-edit.md) |\n| Qwen Image Edit 2511 | [`bfs_head_v5_2511_original.safetensors`](bfs_head_v5_2511_original.safetensors), [`bfs_head_v5_2511_merged_version_rank_16_fp16.safetensors`](bfs_head_v5_2511_merged_version_rank_16_fp16.safetensors), [`bfs_head_v5_2511_merged_version_rank_32_fp32.safetensors`](bfs_head_v5_2511_merged_version_rank_32_fp32.safetensors) | See the [Qwen Edit guide](docs/qwen-image-edit.md) |\n| Flux 2 Klein | [`bfs_head_v1_flux-klein_4b.safetensors`](bfs_head_v1_flux-klein_4b.safetensors), [`bfs_head_v1_flux-klein_9b_step3500_rank128.safetensors`](bfs_head_v1_flux-klein_9b_step3500_rank128.safetensors), [`bfs_head_v1_flux-klein_9b_step3750_rank64.safetensors`](bfs_head_v1_flux-klein_9b_step3750_rank64.safetensors), [`bfs_head_v1.1_optional_flux-klein_4b.safetensors`](bfs_head_v1.1_optional_flux-klein_4b.safetensors) | See the [Flux guide](docs/flux-2-klein.md) |\n| Krea 2 | [`bfs_head_swap_v1.1_krea2.safetensors`](bfs_head_swap_v1.1_krea2.safetensors), [`bfs_body_swap_v1_krea2.safetensors`](bfs_body_swap_v1_krea2.safetensors) | See the [Krea 2 guide](docs/krea-2.md) |\n| Experimental LTX-2 | [`bfs_head_swap_v1_ltx2_ic_lora_5000.safetensors`](bfs_head_swap_v1_ltx2_ic_lora_5000.safetensors), [`bfs_head_swap_v1_ltx2_ic_lora_6000.safetensors`](bfs_head_swap_v1_ltx2_ic_lora_6000.safetensors), [`bfs_head_swap_v1_ltx2_ic_lora_12000_train_3.safetensors`](bfs_head_swap_v1_ltx2_ic_lora_12000_train_3.safetensors) | See the [LTX-2 guide](docs/ltx-2.md) |\n\n## General rules\n\n- Use the LoRA with its matching base model whenever possible.\n- Follow the input order in the version guide; several BFS head releases intentionally use body\n  first and reference face second.\n- Start at LoRA strength `1.0`, then adjust for the base model, resolution, and reference image.\n- The examples and workflows in this repository are demonstrations, not a guarantee of identical\n  results on every image.\n- Do not use or share results involving public figures or people who have not given consent.\n\n## Examples\n\nThe existing examples are grouped by workflow family in the individual guides. Qwen Image 2.1\nexamples are `images/bfs_qwen_2.1_1.png` through `images/bfs_qwen_2.1_8.png`.\n\n## Support\n\nFeel free to experiment and share your results responsibly. For workflow issues, first update\nComfyUI and verify that the base model, text encoder, VAE, workflow, and LoRA all belong to the\nsame model family.\n', 'seedvr2-APACHE.txt': '                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "{}"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright 2025 seed\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'qwen3-4b-APACHE.txt': '\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright 2024 Alibaba Cloud\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.', 'bfs-klein-guide.md': 'Source: https://huggingface.co/Alissonerdx/BFS-Best-Face-Swap/blob/2209ae537ac5e883d6ec7db5b7bd6dccf1a0eba7/docs/flux-2-klein.md\n\n# Flux 2 Klein Head-Swap LoRAs\n\nThese BFS weights target Flux 2 Klein 4B/9B variants. They use the body-first input contract:\n\n- **Image 1:** target body/base image\n- **Image 2:** reference face/head\n\nDo not reverse the images for these releases.\n\n## Downloads\n\n- [`bfs_head_v1_flux-klein_4b.safetensors`](../bfs_head_v1_flux-klein_4b.safetensors)\n- [`bfs_head_v1_flux-klein_9b_step3500_rank128.safetensors`](../bfs_head_v1_flux-klein_9b_step3500_rank128.safetensors)\n- [`bfs_head_v1_flux-klein_9b_step3750_rank64.safetensors`](../bfs_head_v1_flux-klein_9b_step3750_rank64.safetensors)\n- [`bfs_head_v1.1_optional_flux-klein_4b.safetensors`](../bfs_head_v1.1_optional_flux-klein_4b.safetensors)\n\n## Prompt\n\n```text\nhead_swap: start with Picture 1 as the base image, keeping its lighting, environment, and background. remove the head from Picture 1 completely and replace it with the head from Picture 2, strictly preserving the hair, eye color, and nose structure of Picture 2. copy the direction of the eye, head rotation, and micro-expressions from Picture 1. Keep the body, clothing, pose, framing, and background intact.\n```\n\nYou can add a short expression or pose instruction when it is visible in the target image. Keep\nthe edit focused on the head and avoid requesting an unrelated scene change in the same prompt.\n\n## Workflow\n\n[`workflows/Head Swap V1 Flux 2 Klein 4b_9b (base_distill).json`](../workflows/Head%20Swap%20V1%20Flux%202%20Klein%204b_9b%20(base_distill).json)\n\nStart around LoRA strength `1.0`; adjust after checking identity preservation, hair boundaries,\nexpression transfer, and skin blending.\n\n[Back to the repository index](../README.md)\n', 'mediapipe-source-APACHE.txt': '                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n\n===========================================================================\nFor files under tasks/cc/text/language_detector/custom_ops/utils/utf/\n===========================================================================\n/*\n * The authors of this software are Rob Pike and Ken Thompson.\n *              Copyright (c) 2002 by Lucent Technologies.\n * Permission to use, copy, modify, and distribute this software for any\n * purpose without fee is hereby granted, provided that this entire notice\n * is included in all copies of any software which is or includes a copy\n * or modification of this software and in all copies of the supporting\n * documentation for such software.\n * THIS SOFTWARE IS BEING PROVIDED "AS IS", WITHOUT ANY EXPRESS OR IMPLIED\n * WARRANTY.  IN PARTICULAR, NEITHER THE AUTHORS NOR LUCENT TECHNOLOGIES MAKE ANY\n * REPRESENTATION OR WARRANTY OF ANY KIND CONCERNING THE MERCHANTABILITY\n * OF THIS SOFTWARE OR ITS FITNESS FOR ANY PARTICULAR PURPOSE.\n */\n'}
if not LICENSE_FILES and '__file__' in globals():
 _license_root=Path(__file__).with_name('licenses')
 if _license_root.is_dir():LICENSE_FILES={p.name:p.read_text() for p in _license_root.iterdir() if p.is_file()}

def file_sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
 return h.hexdigest()

def selected_assets(pack):
 if pack not in ('all','create_edit','finish'):raise ValueError('Use all, create_edit or finish')
 return [(a['folder'],a['name'],a['source'],a['sha256'],a['bytes']) for a in CATALOG['assets'] if pack=='all' or a['pack']==pack]

def validate_manifest(manifest):
 if manifest.get('profile')!='commercial-v7' or not manifest.get('complete'):raise RuntimeError('Not a complete commercial-v7 pack')
 expected=selected_assets(manifest.get('pack'))
 records=manifest.get('files',[])
 wanted={(n,z,d) for _,n,_,d,z in expected}
 actual={(a.get('path'),a.get('bytes'),a.get('sha256')) for a in records}
 if wanted!=actual or len(records)!=len(wanted):raise RuntimeError('Manifest is not the approved commercial model allowlist')
 return True

def reject_extra_models(root,pack):
 allowed={a[1] for a in selected_assets(pack)}
 for p in Path(root).rglob('*'):
  if p.suffix.lower() in ('.onnx','.zip','.pth','.pt','.safetensors','.task','.bin') and (p.parent!=Path(root) or p.name not in allowed):raise RuntimeError('Unapproved extra model in pack: '+p.name)

def transfer(url,target,digest,size):
 target=Path(target);target.parent.mkdir(parents=True,exist_ok=True)
 if target.exists():
  if target.stat().st_size==size and file_sha(target)==digest:
   print('Verified; reused:',target.name,flush=True);return
  raise RuntimeError('Existing file failed verification; preserve or move it before retrying: '+str(target))
 partial=target.with_name(target.name+'.partial')
 for attempt in range(3):
  offset=partial.stat().st_size if partial.exists() else 0
  if offset>=size:
   if offset==size and file_sha(partial)==digest:partial.replace(target);return
   partial.unlink();offset=0
  if shutil.disk_usage(target.parent).free<size-offset+512*1024**2:raise RuntimeError('Insufficient free disk for '+target.name)
  req=urllib.request.Request(url,headers={'Range':f'bytes={offset}-'} if offset else {})
  try:
   with urllib.request.urlopen(req,timeout=120) as response:
    resume=offset>0 and response.status==206
    if resume and not response.headers.get('Content-Range','').startswith(f'bytes {offset}-'):raise RuntimeError('Unexpected range response')
    written=offset if resume else 0;reported=written
    with open(partial,'ab' if resume else 'wb') as f:
     while True:
      data=response.read(8*1024*1024)
      if not data:break
      f.write(data);written+=len(data)
      if written-reported>=128*1024**2:
       print(f'{target.name}: {written/size:.0%}',flush=True);reported=written
   if partial.stat().st_size!=size or file_sha(partial)!=digest:raise RuntimeError('Incomplete download or checksum mismatch')
   partial.replace(target);return
  except (OSError,urllib.error.URLError,RuntimeError) as e:
   if attempt==2:raise RuntimeError('Download failed for '+target.name+'; rerun to retry its partial file.') from e
   print('Retrying',target.name,type(e).__name__,flush=True);time.sleep(2)

def build_pack(pack,output_root=Path('/kaggle/working/image-studio-model-packs'),input_root=Path('/kaggle/input')):
 assets=selected_assets(pack)
 root=Path(output_root)/pack;root.mkdir(parents=True,exist_ok=True)
 lock=open(root/'.builder.lock','w')
 try:fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
 except BlockingIOError:lock.close();raise RuntimeError('This pack is already being built.')
 try:
  manifest=root/'manifest.json'
  # An interrupted rebuild must never leave a completed manifest behind.
  if manifest.exists():manifest.replace(root/'manifest.previous.json')
  print('Building',pack,'on CPU; model bytes:',sum(a[4] for a in assets),flush=True)
  reject_extra_models(root,pack)
  required=sum(size for _,name,_,_,size in assets if not (root/name).exists())
  if shutil.disk_usage(root).free<required+512*1024**2:raise RuntimeError('Insufficient disk for the complete '+pack+' pack plus 512 MiB reserve. Existing files are retained; no downloads started.')
  records=[]
  for folder,name,url,digest,size in assets:
   target=root/name;target.parent.mkdir(parents=True,exist_ok=True)
   # Saved packs must contain real files: input symlinks would not be a durable output.
   if not target.exists() and Path(input_root).exists():
    for candidate in Path(input_root).rglob(name):
     if candidate.is_file() and candidate.stat().st_size==size and file_sha(candidate)==digest:
      if shutil.disk_usage(target.parent).free<size+512*1024**2:raise RuntimeError('Insufficient disk to copy saved model')
      tmp=target.with_name(name+'.partial');shutil.copyfile(candidate,tmp);tmp.replace(target);print('Copied saved input:',name,flush=True);break
   transfer(url,target,digest,size)
   if target.is_symlink() or target.stat().st_size!=size or file_sha(target)!=digest:raise RuntimeError('Pack verification failed: '+name)
   records.append({'path':target.relative_to(root).as_posix(),'bytes':size,'sha256':digest,'source':url})
  data={'schema':2,'profile':'commercial-v7','pack':pack,'complete':True,'model_bytes':sum(a[4] for a in assets),'files':records}
  validate_manifest(data)
  data['license_catalog']=CATALOG
  for name,content in LICENSE_FILES.items():(root/name).write_text(content)
  tmp=manifest.with_suffix('.json.partial');tmp.write_text(json.dumps(data,indent=2));tmp.replace(manifest)
  print('\nPACK COMPLETE:',root,flush=True)
  print('Save Version with outputs. In Image Studio Commercial v7: Add Input → Notebook Output Files → select this saved version.',flush=True)
  print('No GPU or image generation was used. Keep model packs private and follow model licenses.',flush=True)
  return data
 finally:fcntl.flock(lock,fcntl.LOCK_UN);lock.close()

def dataset_identity(owner,manifest):
 import re
 if not isinstance(owner,str) or not re.fullmatch(r'[A-Za-z0-9_-]+',owner):raise ValueError('Set DATASET_OWNER to your Kaggle username or use configured Kaggle credentials.')
 fingerprint=hashlib.sha256(json.dumps(manifest['files'],sort_keys=True).encode()).hexdigest()[:12]
 # Local pack names may have underscores; Kaggle dataset URL slugs may not.
 pack_slug=re.sub(r'[^a-z0-9]+','-',manifest['pack'].lower()).strip('-')
 slug='image-studio-commercial-'+pack_slug+'-'+fingerprint
 if not 6<=len(slug)<=50 or not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*',slug):raise ValueError('Invalid Kaggle Dataset slug; no files were uploaded.')
 return owner+'/'+slug,fingerprint

def kaggle_error_text(error):
 """Keep the actionable reason without exposing credentials or response headers."""
 import re
 message=str(error)
 response=getattr(error,'response',None)
 if response is not None:
  try:
   data=response.json()
   if isinstance(data,dict):message=str(data.get('message') or data.get('error') or data.get('detail') or message)
  except (ValueError,TypeError):pass
 for key in ('KAGGLE_API_TOKEN','KAGGLE_KEY'):
  value=os.environ.get(key)
  if value:message=message.replace(value,'[REDACTED]')
 message=re.sub(r'(?i)Bearer\s+[^\s,;]+','Bearer [REDACTED]',message)
 message=re.sub(r'(?i)((?:api[_-]?key|access_token|token|authorization)\s*[=:]\s*)[^\s,;]+',r'\1[REDACTED]',message)
 return message[:1800]

def authenticated_api():
 # Kaggle API uses private notebook credentials. Never print or store their values.
 if not os.environ.get('KAGGLE_API_TOKEN') and not os.environ.get('KAGGLE_KEY'):
  try:
   from kaggle_secrets import UserSecretsClient
   os.environ['KAGGLE_API_TOKEN']=UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
  except Exception:pass
 if not os.environ.get('KAGGLE_API_TOKEN') and not os.environ.get('KAGGLE_KEY'):
  try:
   from kaggle_secrets import UserSecretsClient
   secrets=UserSecretsClient()
   os.environ['KAGGLE_USERNAME']=secrets.get_secret('KAGGLE_USERNAME')
   os.environ['KAGGLE_KEY']=secrets.get_secret('KAGGLE_KEY')
  except Exception:pass
 from kaggle.api.kaggle_api_extended import KaggleApi
 api=KaggleApi();api.authenticate()
 return api

def save_private_dataset(pack_root,owner='',api=None):
 root=Path(pack_root);manifest=json.loads((root/'manifest.json').read_text())
 validate_manifest(manifest)
 reject_extra_models(root,manifest['pack'])
 for name in LICENSE_FILES:
  if not (root/name).is_file() or (root/name).read_text()!=LICENSE_FILES[name]:raise RuntimeError('Missing/changed license notice: '+name)
 for record in manifest['files']:
  p=root/record['path']
  if p.is_symlink() or not p.is_file() or p.stat().st_size!=record['bytes'] or file_sha(p)!=record['sha256']:raise RuntimeError('Model verification failed before Dataset upload: '+p.name)
 if api is None:api=authenticated_api()
 owner=owner or api.config_values.get('username','')
 handle,fingerprint=dataset_identity(owner,manifest)
 url='https://www.kaggle.com/datasets/'+handle
 receipt=root/'dataset-upload.json'
 if receipt.exists():
  previous=json.loads(receipt.read_text())
  if previous.get('handle')==handle and previous.get('submitted') and previous.get('manifest_fingerprint')==fingerprint:
   print('Previously submitted Dataset:',url,flush=True);return url
 # Match the previously working builder: call create_new directly.
 # Do not gate creation on our own dataset_status/datasets.get request.
 # Kaggle can deny lookup on a not-yet-created/private Dataset; that does not
 # establish that creation is forbidden. create_new never versions/overwrites.
 metadata={'title':'Image Studio '+manifest['pack']+' '+fingerprint,'id':handle,'licenses':[{'name':'other'}],'description':'Private verified model pack for Image Studio. Original model licenses apply; no relicensing. See manifest.json for model sources and SHA256 values.'}
 (root/'dataset-metadata.json').write_text(json.dumps(metadata,indent=2))
 # Stage hardlinks to the exact pack files, avoiding an extra multi-GB copy.
 # Do not upload builder locks, old manifests, receipts or failure diagnostics.
 import tempfile
 with tempfile.TemporaryDirectory(prefix='.studio-upload-',dir=root.parent) as tmp:
  stage=Path(tmp)
  for name in [r['path'] for r in manifest['files']]+list(LICENSE_FILES)+['manifest.json','dataset-metadata.json']:
   os.link(root/name,stage/name)
  result=api.dataset_create_new(str(stage),public=False,quiet=False,convert_to_csv=False,dir_mode='skip')
 field=lambda key:result.get(key) if isinstance(result,dict) else getattr(result,key,None)
 error=field('error') or field('error_message')
 status=field('status')
 if error or (status and str(status).lower() in ('error','failed','failure')):
  raise RuntimeError('Kaggle rejected Dataset creation: '+kaggle_error_text(error or status)+'. Verified models are retained; no model downloads are needed.')
 if result is None:raise RuntimeError('Kaggle returned no creation response. Verified models are retained. Check the Dataset page before retrying.')
 receipt.write_text(json.dumps({'handle':handle,'url':url,'submitted':True,'requested_private':True,'manifest_fingerprint':fingerprint},indent=2))
 print('PRIVATE DATASET SUBMITTED:',url,flush=True)
 print('Wait for Kaggle processing, then Image Studio Commercial v7 → Add Input → Your Datasets. No PC download needed.',flush=True)
 return url


def publish_pack(pack_root,owner='',api=None):
 try:return save_private_dataset(pack_root,owner,api)
 except (Exception,SystemExit) as error:
  detail=kaggle_error_text(error)
  root=Path(pack_root)
  (root/'dataset-error.json').write_text(json.dumps({'stage':'dataset-publication','reason':detail,'models_retained':True},indent=2))
  raise RuntimeError('Dataset publication failed: '+detail+' Save Version with outputs before closing this session. Rerunning reuses verified weights. Diagnostic: '+str(root/'dataset-error.json')) from None

def create_complete_dataset(output_root=Path('/kaggle/working/image-studio-model-packs'),input_root=Path('/kaggle/input'),owner='',api=None):
 """Use one native create_new request; feed its uploader one verified file at a time."""
 import inspect,tempfile
 root=Path(output_root)/'all';root.mkdir(parents=True,exist_ok=True)
 lock=open(root/'.builder.lock','w')
 try:fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
 except BlockingIOError:lock.close();raise RuntimeError('The ALL MODELS Dataset creator is already running.')
 original_upload=None
 try:
  assets=selected_assets('all')
  records=[dict(path=n,bytes=z,sha256=d,source=u) for _,n,u,d,z in assets]
  manifest=dict(schema=2,profile='commercial-v7',pack='all',complete=True,model_bytes=sum(a[4] for a in assets),files=records,license_catalog=CATALOG)
  validate_manifest(manifest)
  if api is None:api=authenticated_api()
  handle,fingerprint=dataset_identity(owner or api.config_values.get('username',''),manifest)
  url='https://www.kaggle.com/datasets/'+handle;receipt=root/'dataset-upload.json'
  if receipt.exists():
   old=json.loads(receipt.read_text())
   if old.get('handle')==handle and old.get('submitted') and old.get('manifest_fingerprint')==fingerprint:
    print('Previously submitted ALL MODELS Dataset:',url,flush=True);return url
  # Locate verified sources before reserving disk. Existing source files are never
  # copied, moved or removed. Only this creator's missing-file cache is released.
  print('Finding verified saved models; existing sources will not be copied or deleted.',flush=True)
  sources={}
  for _,name,_,digest,size in assets:
   for search in [Path(input_root),Path(output_root)]:
    if not search.exists():continue
    found=next((q for q in search.rglob(name) if q.is_file() and q.stat().st_size==size and file_sha(q)==digest),None)
    if found:
     sources[name]=found.resolve();print('Reusing existing model:',name,flush=True);break
  cache=Path(output_root)/'.all-models-download-cache'
  if cache.is_symlink():raise RuntimeError('Temporary download cache must not be a symlink.')
  cache.mkdir(exist_ok=True)
  missing=[a for a in assets if a[1] not in sources]
  peak=max((a[4] for a in missing),default=0)
  free=shutil.disk_usage(cache).free
  print('ONE ALL MODELS Dataset:',len(assets),'models;',sum(a[4] for a in assets),'bytes total',flush=True)
  print('Verified existing sources:',len(sources),'; largest missing download:',peak,'bytes; free disk:',free,flush=True)
  if free<peak+(512*1024**2 if peak else 16*1024**2):raise RuntimeError('Need room for only the largest missing model ('+str(peak)+' bytes) plus 512 MiB reserve; free '+str(free)+'. Existing model files were not changed.')
  original_upload=api.upload_files;signature=inspect.signature(original_upload)
  if 'folder' not in signature.parameters:raise RuntimeError('Kaggle uploader has an unsupported folder argument; no downloads started.')
  metadata=dict(title='Image Studio all '+fingerprint,id=handle,licenses=[{'name':'other'}],description='Complete private Image Studio model pack. Original component licenses apply; see manifest.json and included license notices.')
  state=root/'upload-progress.json'
  upload_finished=False
  def stream_upload(*args,**kwargs):
   nonlocal upload_finished
   # dataset_create_new owns its normal request and upload context. Delegate all
   # transfers to its unmodified uploader, changing only each input folder.
   def send(folder):
    bound=signature.bind(*args,**kwargs);bound.arguments['folder']=str(folder)
    return original_upload(*bound.args,**bound.kwargs)
   uploaded=[]
   request=signature.bind(*args,**kwargs).arguments['request']
   for _,name,source_url,digest,size in sorted(assets,key=lambda a:a[1] not in sources):
    source=sources.get(name,cache/name)
    owned=source.parent.resolve()==cache.resolve()
    if owned:transfer(source_url,source,digest,size)
    if not source.is_file() or source.stat().st_size!=size or file_sha(source)!=digest:raise RuntimeError('Model failed upload verification: '+name)
    with tempfile.TemporaryDirectory(prefix='.single-model-',dir=root) as temp:
     # Kaggle's native uploader uses isfile/open, which follow this absolute link.
     # Uploaded Dataset files contain actual bytes, not the local link.
     (Path(temp)/name).symlink_to(source.resolve())
     print('Uploading verified model:',name,flush=True)
     count=len(request.files);send(temp)
     if len(request.files)!=count+1:raise RuntimeError('Kaggle uploader did not confirm the file token: '+name)
    uploaded.append(name);state.write_text(json.dumps({'handle':handle,'uploaded_models':uploaded,'dataset_created':False},indent=2))
    if owned:
     # Only newly downloaded cache files, after a successful native file upload.
     source.unlink();print('Released temporary download:',name,flush=True)
   with tempfile.TemporaryDirectory(prefix='.pack-notices-',dir=root) as temp:
    folder=Path(temp);(folder/'manifest.json').write_text(json.dumps(manifest,indent=2))
    for name,content in LICENSE_FILES.items():(folder/name).write_text(content)
    send(folder)
   upload_finished=True
  api.upload_files=stream_upload
  with tempfile.TemporaryDirectory(prefix='.dataset-request-',dir=root) as temp:
   (Path(temp)/'dataset-metadata.json').write_text(json.dumps(metadata,indent=2))
   result=api.dataset_create_new(temp,public=False,quiet=False,convert_to_csv=False,dir_mode='skip')
  field=lambda k:result.get(k) if isinstance(result,dict) else getattr(result,k,None)
  error=field('error') or field('error_message');status=field('status')
  if result is None or error or str(status).lower() in ('error','failed','failure'):raise RuntimeError('Kaggle Dataset creation was not confirmed: '+kaggle_error_text(error or status or 'empty response'))
  if not upload_finished:raise RuntimeError('Kaggle did not run the complete model upload; no success receipt was written.')
  receipt.write_text(json.dumps(dict(handle=handle,url=url,submitted=True,requested_private=True,manifest_fingerprint=fingerprint,upload_mode='one-model-at-a-time'),indent=2))
  (root/'manifest.json').write_text(json.dumps(manifest,indent=2))
  print('PRIVATE ALL MODELS DATASET SUBMITTED:',url,flush=True)
  print('Wait for processing, then attach this ONE Dataset to Studio v9. No model preparation runs in the studio.',flush=True)
  return url
 except (Exception,SystemExit) as error:
  detail=kaggle_error_text(error)
  (root/'dataset-error.json').write_text(json.dumps({'stage':'single-dataset-creation','reason':detail,'existing_sources_untouched':True,'temporary_uploaded_downloads_may_be_released':True},indent=2))
  raise RuntimeError(detail+' Existing source models are untouched. Dataset creation has not been confirmed; see '+str(root/'dataset-error.json')) from None
 finally:
  if original_upload is not None:api.upload_files=original_upload
  fcntl.flock(lock,fcntl.LOCK_UN);lock.close()

if __name__=='__main__':
 if AUTO_CREATE_DATASET:create_complete_dataset(owner=DATASET_OWNER)
 else:build_pack(PACK)
